# Building Scene Reconstruction: Video → GLOMAP → Point Cloud → Mesh → Blender

This notebook takes a video walkaround/flythrough of a building and turns it into a textured
mesh you can import into Blender.

**Pipeline**

1. Extract frames from the video with `ffmpeg`
2. Extract SIFT features + match them with COLMAP
3. Run **GLOMAP** for fast global Structure-from-Motion (camera poses + sparse point cloud)
4. Densify with COLMAP's `patch_match_stereo` + `stereo_fusion` (dense point cloud)
5. Mesh the dense point cloud (Poisson reconstruction, via Open3D)
6. Clean + export the mesh as `.obj` / `.ply` for Blender

**Requirements** (install once, see next cell):
- `ffmpeg`
- [COLMAP](https://colmap.github.io/) (built with CUDA recommended for dense stereo, but CPU works)
- [GLOMAP](https://github.com/colmap/glomap) (binary `glomap` on PATH)
- Python: `pycolmap`, `open3d`, `numpy`, `trimesh`

> GLOMAP replaces COLMAP's incremental mapper with a much faster **global** SfM solver. It
> reads/writes COLMAP-format databases and sparse models, so everything downstream (dense
> stereo, meshing) still uses the normal COLMAP toolchain.


## 0. Config — set your paths here

In [ ]:
import os
from pathlib import Path

# ---- EDIT THESE ----
PROJECT_DIR = Path("./building_scan")      # everything will live under here
VIDEO_PATH  = Path("./media/test2.mov")  # your source video

FPS = 2                 # frames per second to extract (2-4 is a good start for SfM)
MAX_IMAGE_SIZE = 2048   # resize longest edge (0 = keep original)

CAMERA_MODEL = "OPENCV" # good default for consumer/phone/drone footage (has distortion params)
SINGLE_CAMERA = True    # set True if all frames came from the same camera/lens

# Also run COLMAP's own incremental `mapper` as a second SfM backend, in addition to GLOMAP's
# global solver (Section 4). It reuses the same feature database, so this costs one extra
# mapping pass. Whichever backend registers more images ends up as `best_model` (Section 4b).
RUN_COLMAP_MAPPER = True

# GPU flags — COLMAP's GPU path requires CUDA (NVIDIA only). Default to CPU-only, since
# most laptops (macOS especially — no CUDA there at all, only Metal, which COLMAP doesn't
# use) and many Linux machines don't have a CUDA-enabled COLMAP build. CPU is slower but
# always works. Only flip these to True if you know you have an NVIDIA GPU + CUDA toolkit
# AND a COLMAP build compiled with CUDA support.
USE_GPU_FEATURES = False
USE_GPU_DENSE = False
# ---------------------

IMAGES_DIR         = PROJECT_DIR / "images"
DB_PATH            = PROJECT_DIR / "database.db"
SPARSE_DIR         = PROJECT_DIR / "sparse"          # GLOMAP output (COLMAP sparse model format)
SPARSE_DIR_COLMAP  = PROJECT_DIR / "sparse_colmap"    # COLMAP incremental mapper output (alt SfM backend)
DENSE_DIR          = PROJECT_DIR / "dense"           # COLMAP dense workspace
MESH_DIR           = PROJECT_DIR / "mesh"

for d in [PROJECT_DIR, IMAGES_DIR, SPARSE_DIR, SPARSE_DIR_COLMAP, DENSE_DIR, MESH_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR.resolve())

# --- Environment helper ---
# If this notebook's kernel is running inside a conda/Anaconda environment, conda injects
# environment variables (PATH, CPATH, LIBRARY_PATH, DYLD_*) pointing at its own copies of
# libraries like gflags/glog. If those were present when `cmake`/`ninja` built GLOMAP, the
# linker may have pulled in conda's copy of a library for one dependency and Homebrew's for
# another — baking a permanent conflict into the binary itself (error: "flag 'flagfile' was
# defined more than once"). Stripping DYLD_* helps at *runtime*; stripping PATH/CPATH/
# LIBRARY_PATH matters at *build* time. Use clean_env() for every cmake/ninja/colmap/glomap
# subprocess call, and rebuild from a clean state if a prior build happened without this.
_ENV_VARS_TO_STRIP = [
    "DYLD_LIBRARY_PATH", "DYLD_FALLBACK_LIBRARY_PATH", "DYLD_INSERT_LIBRARIES",
    "CPATH", "C_INCLUDE_PATH", "CPLUS_INCLUDE_PATH", "LIBRARY_PATH", "LD_LIBRARY_PATH",
    "CMAKE_PREFIX_PATH", "PYTHONPATH",
]

def clean_env():
    env = os.environ.copy()
    for var in _ENV_VARS_TO_STRIP:
        env.pop(var, None)

    # Remove any conda/anaconda directories from PATH so its cmake/compilers/libs can't be
    # picked up ahead of Homebrew's.
    path_parts = env.get("PATH", "").split(os.pathsep)
    path_parts = [p for p in path_parts if "conda" not in p.lower()]
    env["PATH"] = os.pathsep.join(path_parts)

    return env


## 1. Install dependencies

Run once. If you're on a machine that already has COLMAP/GLOMAP built, skip the apt/build
lines and just make sure `colmap` and `glomap` are on your `PATH`.


In [ ]:
# --- Python deps ---
%pip install pycolmap open3d numpy trimesh opencv-python tqdm -q


In [ ]:
# --- System deps ---
# This cell auto-detects Linux (apt) vs macOS (brew). GLOMAP has no package manager formula
# on either platform, so it's always built from source below.
import platform, shutil

system = platform.system()
print("Detected OS:", system)

if system == "Linux" and shutil.which("apt-get"):
    !apt-get -qq update && apt-get -qq install -y ffmpeg colmap cmake ninja-build \
        libboost-all-dev libeigen3-dev libflann-dev libfreeimage-dev libmetis-dev \
        libgoogle-glog-dev libgflags-dev libsqlite3-dev libglew-dev qtbase5-dev \
        libqt5opengl5-dev libcgal-dev libceres-dev
elif system == "Darwin":
    # macOS: Homebrew. If you don't have Homebrew: https://brew.sh
    # Note: no Qt here — GLOMAP is CLI-only and doesn't need it. libomp is required because
    # AppleClang has no built-in OpenMP support (unlike GCC/Linux Clang), and GLOMAP/COLMAP's
    # CMake looks for it explicitly. opencv is required by the `colmap` binary itself at
    # runtime (e.g. for line-segment detection) — normally pulled in automatically as a
    # colmap dependency, but installed explicitly here in case it's missing/unlinked.
    !brew install ffmpeg colmap cmake ninja boost eigen flann freeimage \
        metis glog gflags cgal ceres-solver libomp opencv
else:
    print("Unrecognized OS — install ffmpeg + colmap manually: https://colmap.github.io/install.html")


In [ ]:
# --- Build GLOMAP from source (no prebuilt package on Linux or macOS) ---
# FORCE_REBUILD: flip to True once if a previous build got poisoned by conda's libraries
# (e.g. you saw "flag 'flagfile' was defined more than once" mentioning both a Homebrew path
# and a conda/anaconda 'croot' path). That means CMake linked gflags/glog from two different
# places at build time — no runtime env var can undo that, the build itself must be redone
# cleanly. Leave this False normally: the check below already verifies the existing binary
# actually runs before reusing it, so forcing True here means recompiling GLOMAP *and* its
# vendored COLMAP from scratch (several minutes) on every single notebook run.
FORCE_REBUILD = False

import shutil, subprocess, os, platform

build_dir = PROJECT_DIR.parent / "glomap_src"

def _glomap_binary_works(path: str) -> bool:
    try:
        result = subprocess.run([path, "-h"], capture_output=True, text=True, env=clean_env())
        return "Global Structure-from-Motion" in (result.stdout + result.stderr)
    except OSError:
        return False

existing_glomap = shutil.which("glomap", path=clean_env()["PATH"])

if FORCE_REBUILD and build_dir.exists():
    print("Removing previous build directory (FORCE_REBUILD=True)...")
    shutil.rmtree(build_dir)
    existing_glomap = None

if existing_glomap and not FORCE_REBUILD and _glomap_binary_works(existing_glomap):
    print("glomap already on PATH and working:", existing_glomap)
else:
    if not build_dir.exists():
        subprocess.run(
            ["git", "clone", "https://github.com/colmap/glomap.git", str(build_dir)],
            check=True, env=clean_env(),
        )

    cmake_build = build_dir / "build"
    cmake_build.mkdir(exist_ok=True)

    # -DCMAKE_IGNORE_PREFIX_PATH tells CMake to never resolve find_package() into conda's
    # prefix at all, even if some conda env var slips through — belt and suspenders on top
    # of clean_env() already stripping conda from PATH/CPATH/LIBRARY_PATH.
    cmake_cmd = [
        "cmake", "..", "-GNinja",
        "-DCMAKE_IGNORE_PREFIX_PATH=/opt/anaconda3;/opt/miniconda3;/opt/conda",
    ]
    if platform.system() == "Linux":
        cmake_cmd += ["-DCMAKE_CUDA_ARCHITECTURES=native"]
    else:
        # No CUDA on macOS.
        cmake_cmd += ["-DGLOMAP_CUDA_ENABLED=OFF", "-DCUDA_ENABLED=OFF"]

        # AppleClang has no built-in OpenMP — point CMake at Homebrew's libomp explicitly,
        # since it's keg-only (not symlinked into /opt/homebrew by default).
        libomp_prefix = subprocess.run(
            ["brew", "--prefix", "libomp"],
            capture_output=True, text=True, check=True, env=clean_env(),
        ).stdout.strip()
        omp_include = f"{libomp_prefix}/include"
        omp_lib = f"{libomp_prefix}/lib/libomp.dylib"
        cmake_cmd += [
            f"-DOpenMP_C_FLAGS=-Xpreprocessor -fopenmp -I{omp_include}",
            f"-DOpenMP_C_LIB_NAMES=omp",
            f"-DOpenMP_CXX_FLAGS=-Xpreprocessor -fopenmp -I{omp_include}",
            f"-DOpenMP_CXX_LIB_NAMES=omp",
            f"-DOpenMP_omp_LIBRARY={omp_lib}",
        ]

    result = subprocess.run(cmake_cmd, cwd=cmake_build, capture_output=True, text=True, env=clean_env())
    print(result.stdout)
    print(result.stderr)
    result.check_returncode()  # raises with the message already printed above if it failed
    build_result = subprocess.run(["ninja"], cwd=cmake_build, capture_output=True, text=True, env=clean_env())
    print(build_result.stdout[-5000:])  # tail — full logs can be long
    print(build_result.stderr[-5000:])
    build_result.check_returncode()

    # Skip `sudo ninja install` — Jupyter can't prompt for a password. The binary already
    # exists right where it was built; just put that folder on PATH for this notebook.
    built_binary_dir = cmake_build / "glomap"
    assert (built_binary_dir / "glomap").exists(), "glomap binary not found after build"
    os.environ["PATH"] = f"{built_binary_dir}:{os.environ['PATH']}"

    print("glomap installed:", shutil.which("glomap"))


In [ ]:
# --- Build the exact COLMAP binary GLOMAP vendors (this is what fixes the SIGABRT below) ---
# GLOMAP does NOT use your system `colmap` for anything internally — it FetchContent's its
# own pinned COLMAP commit (see glomap_src/thirdparty/CMakeLists.txt) and statically links
# against THAT library's SQLite database reader. If the .db file you feed to `glomap mapper`
# was written by a *different* COLMAP build (e.g. Homebrew's 4.1.0), the on-disk schema can
# differ — COLMAP added rig/frame tables to the database after 4.1.0 — and glomap's bundled
# reader aborts with "SQLite error: SQL logic error" / SIGABRT trying to read a shape it
# doesn't recognize. The fix is to build this exact binary once and use it (not `colmap` from
# Homebrew) for every step that writes to database.db — see Section 3 below.
import subprocess

cmake_build = build_dir / "build"

build_colmap = subprocess.run(
    ["ninja", "colmap"], cwd=cmake_build, capture_output=True, text=True, env=clean_env()
)
print(build_colmap.stdout[-3000:])
print(build_colmap.stderr[-3000:])
build_colmap.check_returncode()

matched_colmap_bin = cmake_build / "_deps" / "colmap-build" / "src" / "colmap" / "exe" / "colmap"
assert matched_colmap_bin.exists(), f"expected binary not found at {matched_colmap_bin}"
print("Built matching colmap binary (schema-compatible with glomap) at:", matched_colmap_bin)


> **Note on `glomap` and PATH:** the cell above adds GLOMAP's build output directory to
> `PATH` for *this Python process only* — it won't persist to a new terminal or a restarted
> kernel. That's fine as long as you run this notebook top-to-bottom in one session.
>
> If you'd rather have `glomap` available everywhere (e.g. to use outside this notebook),
> install it properly from a real terminal (not a notebook cell, since that can prompt for
> your password interactively):
> ```bash
> cd <path from the build cell above>/build
> sudo ninja install
> ```
> Or skip `sudo` entirely and install to somewhere on your `PATH` you already own, e.g.
> `~/.local/bin`:
> ```bash
> cmake --install . --prefix ~/.local/bin/..  # or just copy the binary:
> cp glomap/glomap ~/.local/bin/
> ```


In [ ]:
# Sanity check versions (using clean_env() to avoid conda/Homebrew library clashes)
import subprocess

def check_version(cmd):
    result = subprocess.run(cmd, capture_output=True, text=True, env=clean_env())
    out = (result.stdout or result.stderr).strip().splitlines()
    print(out[0] if out else f"(no output, exit code {result.returncode})")

check_version(["ffmpeg", "-version"])
check_version(["colmap", "-h"])                 # system colmap — only used for undistort/dense steps
check_version([str(matched_colmap_bin), "-h"])   # GLOMAP's vendored colmap — used for all DB writes
check_version(["glomap", "-h"])


## 2. Extract frames from the video with ffmpeg

Uses a constant fps sample. For building walkarounds, 2-4 fps with decent camera motion
usually gives COLMAP/GLOMAP enough overlap between consecutive frames (aim for ~60-80%
visual overlap) without drowning the matcher in near-duplicate images.


In [ ]:
import subprocess

def extract_frames(video_path: Path, out_dir: Path, fps: int, max_size: int = 0):
    out_dir.mkdir(parents=True, exist_ok=True)
    # clear old frames so re-runs don't mix frame sets
    for f in out_dir.glob("*.jpg"):
        f.unlink()

    vf_filters = [f"fps={fps}"]
    if max_size and max_size > 0:
        # scale longest edge down to max_size, keep aspect ratio, only if larger
        vf_filters.append(
            f"scale='if(gt(iw,ih),min({max_size},iw),-2)':'if(gt(iw,ih),-2,min({max_size},ih))'"
        )
    vf = ",".join(vf_filters)

    cmd = [
        "ffmpeg", "-y",
        "-i", str(video_path),
        "-vf", vf,
        "-qscale:v", "2",           # high JPEG quality (2 = near-lossless, lower is better)
        str(out_dir / "frame_%05d.jpg"),
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)

    n = len(list(out_dir.glob("*.jpg")))
    print(f"Extracted {n} frames -> {out_dir}")
    return n

n_frames = extract_frames(VIDEO_PATH, IMAGES_DIR, FPS, MAX_IMAGE_SIZE)
assert n_frames > 0, "No frames extracted — check VIDEO_PATH"


### Optional: blur filtering

Motion-blurred frames hurt feature matching. This drops the blurriest frames using a
Laplacian-variance sharpness score.


In [ ]:
import cv2
import numpy as np

def sharpness_score(img_path: Path) -> float:
    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return 0.0
    return cv2.Laplacian(img, cv2.CV_64F).var()

def filter_blurry_frames(img_dir: Path, keep_fraction: float = 0.9):
    frames = sorted(img_dir.glob("*.jpg"))
    scores = [(f, sharpness_score(f)) for f in frames]
    scores.sort(key=lambda x: x[1])  # ascending: blurriest first
    n_drop = int(len(scores) * (1 - keep_fraction))
    dropped = scores[:n_drop]
    for f, s in dropped:
        f.unlink()
    print(f"Dropped {len(dropped)} blurry frames, kept {len(frames) - len(dropped)}")

# Uncomment to enable (drops the blurriest 10% of frames):
# filter_blurry_frames(IMAGES_DIR, keep_fraction=0.9)


## 3. COLMAP: feature extraction + matching

GLOMAP consumes a COLMAP database, so we still use `colmap feature_extractor` and a matcher
to populate it. `sequential_matcher` is a good fit for video frames (it matches each frame to
its temporal neighbors, plus optional loop-closure checks), and is much faster than exhaustive
matching for long walkthroughs.


In [ ]:
def run(cmd):
    print("Running:", " ".join(cmd))
    result = subprocess.run(cmd, env=clean_env(), capture_output=True, text=True)
    print(result.stdout)
    print(result.stderr)
    result.check_returncode()

# --- Feature extraction ---
# Uses matched_colmap_bin (built alongside GLOMAP in Section 1), NOT the system `colmap` —
# so the database this writes has the exact schema GLOMAP's bundled reader expects. See the
# note on the "Build the exact COLMAP binary..." cell for why this matters.
# Note: a COLMAP build without CUDA support doesn't even register the *.use_gpu flags —
# passing them (even as "0") is a hard "unrecognised option" error, not a no-op. So we only
# add the flag when GPU is actually requested.
extract_cmd = [
    str(matched_colmap_bin), "feature_extractor",
    "--database_path", str(DB_PATH),
    "--image_path", str(IMAGES_DIR),
    "--ImageReader.camera_model", CAMERA_MODEL,
    "--ImageReader.single_camera", "1" if SINGLE_CAMERA else "0",
]
if USE_GPU_FEATURES:
    extract_cmd += ["--SiftExtraction.use_gpu", "1"]
run(extract_cmd)


In [ ]:
# --- Feature matching ---
# sequential_matcher: good default for video (temporal order in filenames matters here,
# which is why ffmpeg's zero-padded frame_%05d.jpg naming above is important). Uses
# matched_colmap_bin for the same schema-compatibility reason as feature extraction above.
match_cmd = [
    str(matched_colmap_bin), "sequential_matcher",
    "--database_path", str(DB_PATH),
    "--SequentialMatching.loop_detection", "1",   # helps close the loop on a walkaround
    "--SequentialMatching.overlap", "10",
]
if USE_GPU_FEATURES:
    match_cmd += ["--SiftMatching.use_gpu", "1"]
run(match_cmd)

# If your video pans around and revisits the same facade non-sequentially (e.g. you looped
# the building more than once, or clipped/reordered footage), exhaustive matching is more
# robust at the cost of speed:
# exhaustive_cmd = [str(matched_colmap_bin), "exhaustive_matcher", "--database_path", str(DB_PATH)]
# if USE_GPU_FEATURES:
#     exhaustive_cmd += ["--SiftMatching.use_gpu", "1"]
# run(exhaustive_cmd)


## 4. GLOMAP: global structure-from-motion

This is the step that replaces COLMAP's (slower, incremental) `mapper` with GLOMAP's global
pose-graph solver. Output is a standard COLMAP sparse reconstruction (`cameras.bin`,
`images.bin`, `points3D.bin`) under `SPARSE_DIR/0`.


In [ ]:
import shutil

# GLOMAP appends to whatever's already in SPARSE_DIR — wipe stale output from a previous run
# so we don't mix old and new reconstructions.
if SPARSE_DIR.exists():
    shutil.rmtree(SPARSE_DIR)
SPARSE_DIR.mkdir(parents=True)

glomap_bin = shutil.which("glomap")
assert glomap_bin, "glomap not found on PATH — ensure it was built/installed (see Section 1)"
print("Using glomap binary:", glomap_bin)

run([
    glomap_bin, "mapper",
    "--database_path", str(DB_PATH),
    "--image_path", str(IMAGES_DIR),
    "--output_path", str(SPARSE_DIR),
])


## 4b. (Optional) COLMAP's own incremental mapper — a second SfM backend

Runs COLMAP's classic incremental `mapper` against the same database (already
schema-matched via `matched_colmap_bin`), so no extra feature extraction/matching is
needed. It's slower than GLOMAP's global solver but sometimes registers images GLOMAP
drops. Controlled by `RUN_COLMAP_MAPPER` in the config cell; the model-selection cell
below picks whichever backend registers more images.

In [ ]:
if RUN_COLMAP_MAPPER:
    SPARSE_DIR_COLMAP.mkdir(parents=True, exist_ok=True)
    if any(SPARSE_DIR_COLMAP.iterdir()):
        print(f"COLMAP mapper output already exists in {SPARSE_DIR_COLMAP} - skipping "
              "(delete that folder to force a rerun).")
    else:
        run([
            str(matched_colmap_bin), "mapper",
            "--database_path", str(DB_PATH),
            "--image_path", str(IMAGES_DIR),
            "--output_path", str(SPARSE_DIR_COLMAP),
        ])
else:
    print("RUN_COLMAP_MAPPER = False - skipping COLMAP's incremental mapper.")


In [ ]:
# GLOMAP (and, if enabled, COLMAP's mapper) each write one or more reconstructions (like
# COLMAP mapper does whenever the scene breaks into disconnected components). Gather
# candidates from both backends and use whichever registers the most images.
candidate_dirs = [d for d in SPARSE_DIR.iterdir() if d.is_dir()] if SPARSE_DIR.exists() else []
if RUN_COLMAP_MAPPER and SPARSE_DIR_COLMAP.exists():
    candidate_dirs += [d for d in SPARSE_DIR_COLMAP.iterdir() if d.is_dir()]

sparse_models = sorted(candidate_dirs)
print("Sparse models:", sparse_models)
assert sparse_models, "No sparse model produced - check the matching step / image overlap"

# Use the largest reconstruction (most registered images) across all backends.
def n_registered_images(model_dir: Path) -> int:
    import pycolmap
    rec = pycolmap.Reconstruction(str(model_dir))
    return len(rec.images)

best_model = max(sparse_models, key=n_registered_images)
backend = "COLMAP" if SPARSE_DIR_COLMAP in best_model.parents else "GLOMAP"
print(f"Using model: {best_model} ({backend}) with {n_registered_images(best_model)} registered images")


### Quick look at the sparse point cloud

In [ ]:
import pycolmap

rec = pycolmap.Reconstruction(str(best_model))
print(rec.summary())

sparse_ply = PROJECT_DIR / "sparse_points.ply"
rec.export_PLY(str(sparse_ply))
print("Exported sparse point cloud ->", sparse_ply)


## 5. Dense reconstruction (COLMAP MVS)

The sparse point cloud from SfM is too sparse to mesh well on its own. We'd normally densify
it with COLMAP's multi-view stereo (`patch_match_stereo` + `stereo_fusion`) using the poses
GLOMAP just solved.

> **Important limitation:** COLMAP's `patch_match_stereo` has **no CPU implementation at
> all** — it requires a CUDA-enabled build (NVIDIA GPU). If your `colmap` binary reports
> "without CUDA" (check the version string from the sanity-check cell above), this step
> cannot run, full stop — not slowly, not in a degraded mode, it simply isn't compiled in.
>
> The cell below detects this and falls back automatically to meshing directly from the
> **sparse** point cloud instead. It'll work, but the result will be noticeably sparser and
> less detailed than a true dense mesh — building facades with large flat/textureless areas
> will have visible gaps.
>
> If you want a proper dense mesh without an NVIDIA GPU, the standard alternative is
> [OpenMVS](https://github.com/cdcseacave/openMVS), which does support CPU-only dense
> reconstruction (slower, but real MVS). It consumes a COLMAP sparse model as input, so you
> can feed it `best_model` from this notebook and pick up from there — that's a separate
> install/pipeline outside this notebook's scope, but the sparse model exported below
> (`sparse_ply` and the `best_model` directory) is exactly what it needs.


In [ ]:
dense_workspace = DENSE_DIR

run([
    "colmap", "image_undistorter",
    "--image_path", str(IMAGES_DIR),
    "--input_path", str(best_model),
    "--output_path", str(dense_workspace),
    "--output_type", "COLMAP",
])


In [ ]:
# Detect whether this colmap build actually has CUDA (patch_match_stereo needs it — no
# CPU fallback exists). We check the version banner rather than just trying and catching,
# since a failed run can leave partial workspace files behind.
version_check = subprocess.run(["colmap", "-h"], capture_output=True, text=True, env=clean_env())
colmap_banner = (version_check.stdout + version_check.stderr)
HAS_CUDA_COLMAP = "without CUDA" not in colmap_banner and USE_GPU_DENSE

dense_ply = None

if HAS_CUDA_COLMAP:
    run([
        "colmap", "patch_match_stereo",
        "--workspace_path", str(dense_workspace),
        "--workspace_format", "COLMAP",
        "--PatchMatchStereo.geom_consistency", "true",
        "--PatchMatchStereo.gpu_index", "0",
    ])

    dense_ply = dense_workspace / "fused.ply"
    run([
        "colmap", "stereo_fusion",
        "--workspace_path", str(dense_workspace),
        "--workspace_format", "COLMAP",
        "--input_type", "geometric",
        "--output_path", str(dense_ply),
    ])
    print("Dense point cloud ->", dense_ply)
else:
    print(
        "No CUDA-enabled COLMAP build detected — skipping dense stereo.\n"
        "Falling back to meshing directly from the sparse point cloud "
        f"({sparse_ply}).\n"
        "See the markdown note above for the OpenMVS alternative if you want true dense MVS."
    )


## 6. Mesh the dense point cloud (Open3D)

Poisson surface reconstruction gives a closed, watertight-ish mesh — good for buildings.
We also crop the mesh back to the point cloud's bounding box, since Poisson tends to
extrapolate/balloon past the actual data in low-density regions.


In [ ]:
import open3d as o3d
import numpy as np

# Use the dense point cloud if section 5 produced one; otherwise fall back to the sparse
# point cloud from GLOMAP (see the note in section 5 for why this fallback might trigger).
mesh_input_ply = dense_ply if dense_ply is not None else sparse_ply
print("Meshing from:", mesh_input_ply)

pcd = o3d.io.read_point_cloud(str(mesh_input_ply))
print(pcd)

# --- Clean up ---
pcd, _ = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)

# Estimate normals (required for Poisson reconstruction)
pcd.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))
pcd.orient_normals_consistent_tangent_plane(k=30)

o3d.io.write_point_cloud(str(MESH_DIR / "cleaned_points.ply"), pcd)


In [ ]:
# --- Poisson meshing ---
# Lower depth for the sparse-point-cloud fallback — a high depth on sparse input mostly just
# extrapolates empty space rather than adding real detail.
POISSON_DEPTH = 10 if dense_ply is not None else 8

mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
    pcd, depth=POISSON_DEPTH
)

# Trim low-density (extrapolated / unsupported) regions
densities = np.asarray(densities)
density_threshold = np.quantile(densities, 0.02)  # drop bottom 2% density vertices
vertices_to_remove = densities < density_threshold
mesh.remove_vertices_by_mask(vertices_to_remove)

print(mesh)


In [ ]:
# --- Crop mesh to the point cloud's bounding box (removes Poisson's outward "bubbles") ---
bbox = pcd.get_axis_aligned_bounding_box()
mesh = mesh.crop(bbox)

# --- Clean topology ---
mesh.remove_duplicated_vertices()
mesh.remove_duplicated_triangles()
mesh.remove_degenerate_triangles()
mesh.remove_unreferenced_vertices()
mesh.compute_vertex_normals()

print("Final mesh:", mesh)


### Optional: transfer color from the point cloud to the mesh

Poisson reconstruction can carry input colors through automatically if the input point cloud
has them (COLMAP's fused.ply does). If your mesh came out uncolored for any reason, this cell
nearest-neighbor-samples color from the point cloud instead.


In [ ]:
if not mesh.has_vertex_colors():
    pcd_tree = o3d.geometry.KDTreeFlann(pcd)
    colors = np.zeros((len(mesh.vertices), 3))
    pcd_colors = np.asarray(pcd.colors)
    for i, v in enumerate(np.asarray(mesh.vertices)):
        _, idx, _ = pcd_tree.search_knn_vector_3d(v, 1)
        colors[i] = pcd_colors[idx[0]]
    mesh.vertex_colors = o3d.utility.Vector3dVector(colors)
    print("Transferred vertex colors from point cloud")
else:
    print("Mesh already has vertex colors")


## 7. Export for Blender

`.obj` is the safest bet for full compatibility (Blender's importer handles OBJ + vertex
colors/materials reliably). We also keep a `.ply` since it round-trips vertex colors more
simply if you don't need UVs/materials.


In [ ]:
obj_path = MESH_DIR / "building_mesh.obj"
ply_path = MESH_DIR / "building_mesh.ply"

o3d.io.write_triangle_mesh(str(obj_path), mesh, write_vertex_colors=True)
o3d.io.write_triangle_mesh(str(ply_path), mesh, write_vertex_colors=True)

print("Exported:")
print(" -", obj_path.resolve())
print(" -", ply_path.resolve())


### Importing into Blender

- `File > Import > Wavefront (.obj)` (or `.ply`), select the file above.
- If colors look flat/gray: in the Shading tab, add an **Attribute** node (name `Col`) or
  **Vertex Color** node feeding into your Principled BSDF's Base Color — OBJ/PLY vertex
  colors aren't auto-wired into materials by Blender's importer.
- Scale: COLMAP/GLOMAP reconstructions are in an arbitrary unit scale (not meters) unless you
  used ground control points / known distances. If you need real-world scale, measure a known
  distance in the scene (e.g. a floor-to-floor height) and apply a uniform scale in Blender to
  match.
- The reconstruction's up-axis may not match Blender's Z-up convention — you may need to
  rotate the imported object (commonly -90° on X) depending on how COLMAP's camera-relative
  coordinate frame landed.


## 8. (Optional) Visualize point cloud / mesh inline

In [ ]:
# Quick non-interactive preview render (headless-friendly). For an interactive window,
# use o3d.visualization.draw_geometries([mesh]) instead when running locally with a display.

vis = o3d.visualization.Visualizer()
vis.create_window(visible=False)
vis.add_geometry(mesh)
vis.poll_events()
vis.update_renderer()
vis.capture_screen_image(str(MESH_DIR / "mesh_preview.png"))
vis.destroy_window()

from IPython.display import Image
Image(filename=str(MESH_DIR / "mesh_preview.png"))


## Notes / troubleshooting

- **GLOMAP registers few/no images**: usually a matching problem, not a GLOMAP problem —
  increase `--SequentialMatching.overlap`, lower `FPS` isn't the fix (more overlap between
  frames helps more than more frames), or switch to `exhaustive_matcher` for smaller datasets.
- **Dense reconstruction is very slow**: reduce `MAX_IMAGE_SIZE`, or set
  `PatchMatchStereo.geom_consistency` to `false` for a rougher/faster pass first.
- **Mesh has holes on featureless walls**: textureless facades (flat stucco, glass) give SfM
  little to match on. Consider capturing footage with more oblique angles / texture context,
  or increasing `POISSON_DEPTH` won't fix missing geometry — you need more source views of
  that surface.
- **Alternative to Poisson**: for scenes with very uneven point density, try
  `o3d.geometry.TriangleMesh.create_from_point_cloud_ball_pivoting` instead — it tends to
  hallucinate less but leaves more holes.
